In [1]:
import pynuml
from pynuml.labels.lowe import StandardLabelsLowE
from pynuml.labels.flavor_lowe import LowEFlavorLabels

file = pynuml.io.File("/home/apaudel/NuGraph/data/data_20k/event_20k_00.evt.h5")

semantic_labeller = StandardLabelsLowE(e_thr=0.01)
event_labeller = LowEFlavorLabels()

processor = pynuml.process.HitGraphProducer(
    file=file,
    semantic_labeller=semantic_labeller,
    event_labeller=event_labeller,
    label_vertex=True,
    label_position=False,
    optical=True,
    lower_bound=3,
)

In [2]:
file.read_data(0, 100)
evts = iter(file.build_evt())

name, data = processor(next(evts))
skip = 0

while data is None:
    name, data = processor(next(evts))
    skip += 1

print("first valid graph:", name)
print("skipped:", skip)
print(data)

first valid graph: r20000001_sr1_evt10001
skipped: 0
NuGraphData(
  metadata={
    run=20000001,
    subrun=1,
    event=10001,
  },
  sp={ pos=[22, 3] },
  hit={
    plane=[47],
    pos=[47, 2],
    x=[47, 6],
    id=[47],
    y_semantic=[47],
  },
  evt={
    num_nodes=1,
    y=[1],
    y_vtx=[1, 3],
  },
  particle-truth={ num_nodes=1 },
  ophit={
    pos=[257, 3],
    x=[257, 9],
  },
  flash={
    pos=[1, 3],
    x=[1, 10],
  },
  pmt={
    pos=[93, 2],
    x=[93, 4],
  },
  (hit, delaunay, hit)={ edge_index=[2, 256] },
  (hit, delaunay-planar, hit)={ edge_index=[2, 226] },
  (hit, nexus, sp)={ edge_index=[2, 66] },
  (hit, in, evt)={ edge_index=[2, 47] },
  (sp, in, evt)={ edge_index=[2, 22] },
  (hit, cluster-truth, particle-truth)={ edge_index=[2, 47] },
  (ophit, in, pmt)={ edge_index=[2, 99] },
  (pmt, in, flash)={ edge_index=[2, 93] },
  (flash, in, evt)={ edge_index=[2, 1] },
  (sp, knn, pmt)={ edge_index=[2, 220] }
)


/home/apaudel/NuGraph/pynuml/pynuml/process/hitgraph.py:441: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1780240372025/work/torch/csrc/utils/tensor_numpy.cpp:213.)
  data["pmt"].pos = torch.stack([opdet_pos_y[sum_pe["pmt_channel"].values], opdet_pos_z[sum_pe["pmt_channel"].values]], dim=1)


In [3]:
print("node stores:", data.node_types)
print("edge stores:", data.edge_types)

print("event label:", data["evt"].y)
print("event classes:", event_labeller.labels)

print("hit x:", data["hit"].x.shape)
print("hit pos:", data["hit"].pos.shape)
print("sp pos:", data["sp"].pos.shape)

if "ophit" in data.node_types:
    print("ophit x:", data["ophit"].x.shape)
    print("ophit pos:", data["ophit"].pos.shape)

if "pmt" in data.node_types:
    print("pmt x:", data["pmt"].x.shape)
    print("pmt pos:", data["pmt"].pos.shape)

if "flash" in data.node_types:
    print("flash x:", data["flash"].x.shape)
    print("flash pos:", data["flash"].pos.shape)

node stores: ['metadata', 'sp', 'hit', 'evt', 'particle-truth', 'ophit', 'flash', 'pmt']
edge stores: [('hit', 'delaunay', 'hit'), ('hit', 'delaunay-planar', 'hit'), ('hit', 'nexus', 'sp'), ('hit', 'in', 'evt'), ('sp', 'in', 'evt'), ('hit', 'cluster-truth', 'particle-truth'), ('ophit', 'in', 'pmt'), ('pmt', 'in', 'flash'), ('flash', 'in', 'evt'), ('sp', 'knn', 'pmt')]
event label: tensor([1])
event classes: ('cc_nue', 'es_nue')
hit x: torch.Size([47, 6])
hit pos: torch.Size([47, 2])
sp pos: torch.Size([22, 3])
ophit x: torch.Size([257, 9])
ophit pos: torch.Size([257, 3])
pmt x: torch.Size([93, 4])
pmt pos: torch.Size([93, 2])
flash x: torch.Size([1, 10])
flash pos: torch.Size([1, 3])


In [4]:
ok = 0
skipped = 0
labels = []

file.read_data(0, 100)
evts = iter(file.build_evt())

for evt in evts:
    name, data = processor(evt)

    if data is None:
        skipped += 1
        continue

    ok += 1
    labels.append(int(data["evt"].y.item()))

print("OK:", ok)
print("SKIPPED:", skipped)
print("cc_nue:", labels.count(0))
print("es_nue:", labels.count(1))
print("labels:", labels)

OK: 87
SKIPPED: 12
cc_nue: 46
es_nue: 41
labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
